# Generalization

## 1. What Is Generalization?

### 1.1 IID Assumption

- **Independent:** Samples are independent of one another.
- **Identically distributed:** Samples are drawn from the same distribution $P$.
- Training, validation, test sets are all assumed to be sampled from $P$.

### 1.2 What Is Generalization?

> Generalization is a model's capability to perform well on unseen samples drawn from the same underlying data distribution.

### 1.3 Generalization Error

Generalization error is the expected loss of the trained model on samples drawn from $P$.

$$
Err(f) = \mathbb{E}_{(x,y) \sim P}[\ell(f(x), y)].
$$

However, the true distribution $P$ is unknown. How can we calculate this expectation in practice?

We can use an independent test set to estimate the generalization error:

$$
\widehat{Err}(f) = \mathbb{E}_{x, y \sim {D}_{\text{test}}}[\ell(f(x), y)] = \frac{1}{|{D}_{\text{test}}|} \sum_{x,y \in {D}_{\text{test}}} \ell(f(x), y).
$$

$$
\text{test error} \approx \text{generalization error}
$$

## 2. Why Do We Care about Generalization?

Before answering this question, let's look at a KNN example.

In [ ]:
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

from sklearn import datasets, neighbors

# Make the random splits reproducible
np.random.seed(1234)

In [ ]:
# Load the California housing regression dataset
x, y = datasets.fetch_california_housing(return_X_y=True)

num_instances, num_features = x.shape
print(num_instances, num_features)

# Mean squared error
def loss(y, yh):
    return np.mean((y - yh) ** 2)

In [ ]:
# Use 20% of the data as an unseen test set
n_test = num_instances // 5

# Shuffle the sample indices before splitting
inds = np.random.permutation(num_instances)

x_test, y_test = x[inds[:n_test]], y[inds[:n_test]]
x_train, y_train = x[inds[n_test:]], y[inds[n_test:]]

In [ ]:
K_list = range(1, 100)
err_train, err_test = [], []

for K in K_list:
    model = neighbors.KNeighborsRegressor(n_neighbors=K)
    model.fit(x_train, y_train)

    err_train.append(loss(y_train, model.predict(x_train)))
    err_test.append(loss(y_test, model.predict(x_test)))

> **Pause the video and make a prediction:**
>
> 1. What do you expect the training and test errors to look like when $K=1$?
> 2. As $K$ increases, how do you expect these two errors to change?

In [ ]:
# The test curve is shown here only to illustrate generalization.
# We should not use it to select K in practice.
plt.plot(K_list, err_test, '-', label='test')
plt.plot(K_list, err_train, '-', label='train')
plt.legend()
plt.xlabel('K (number of neighbours)')
plt.ylabel('mean squared error')
plt.show()

| Value of $K$ | Training error | Test error | Model behaviour |
|---|---:|---:|---|
| Small $K$ | Very low | High | **Overfitting** |
| Moderate $K$ | Slightly higher | Lowest | **Best generalization** |
| Very large $K$ | High | High | **Underfitting** |

> **Pause the video and think about this question:**
>
> If we select the best $K$ using training error, will it minimize the test error as well?


We care about generalization because we do not want a model that underfits or overfits. We want to find a model that performs well on unseen data—in other words, a model that generalizes well.

## 3. How Do We Select a Model That Generalizes Well?

### 3.1 Why Do We Need a Validation Set?

- Why Can't We Use the Training Set?
    - Our goal is good performance on unseen data.
    - The lowest training error does not imply the lowest generalization error.

- Why Can't We Use the Test Set?
    - **Test leakage problem:** Selecting a model based on test error makes the test set part of the model-selection process.
    - The test set should be reserved for final evaluation.

- The Role of Each Dataset
    - **Training set:** Train the model.
    - **Validation set:** Select the model and hyperparameters.
    - **Test set:** Final evaluation.

![Training, validation, and test data split](https://raw.githubusercontent.com/mikemikezhu/COMP551_McGill/main/assets/data-split.svg)

### 3.2 How Do We Select a Hyperparameter Using a Validation Set?

1. First, split the data into training, validation, and test sets.
2. Loop over candidate values of the hyperparameter.
3. Train each model using the training set.
4. Compute the validation error.
5. After the loop, select the hyperparameter with the lowest validation error.
6. Combine the training and validation sets.
7. Retrain the model using the selected hyperparameter.
8. Evaluate the final model on the test set.

In [ ]:
# Split the data into train (80%), validation (10%), and test (10%) sets
n_test = num_instances // 10
n_valid = num_instances // 10

inds = np.random.permutation(num_instances)

x_test, y_test = x[inds[:n_test]], y[inds[:n_test]]
x_valid, y_valid = x[inds[n_test:n_test + n_valid]], y[inds[n_test:n_test + n_valid]]
x_train, y_train = x[inds[n_test + n_valid:]], y[inds[n_test + n_valid:]]

# Select K using validation error only
K_list = range(1, 30)
err_valid = []

for K in K_list:
    model = neighbors.KNeighborsRegressor(n_neighbors=K)
    model.fit(x_train, y_train)
    err_valid.append(loss(y_valid, model.predict(x_valid)))

plt.plot(K_list, err_valid, label='validation')
plt.legend()
plt.xlabel('K (number of neighbours)')
plt.ylabel('mean squared error')
plt.show()

best_index = np.argmin(err_valid)
best_K = K_list[best_index]
print(f'Best K: {best_K}')
print(f'Validation error: {err_valid[best_index]:.4f}')

# Retrain on all non-test data
x_train_final = np.concatenate([x_train, x_valid], axis=0)
y_train_final = np.concatenate([y_train, y_valid], axis=0)

final_model = neighbors.KNeighborsRegressor(n_neighbors=best_K)
final_model.fit(x_train_final, y_train_final)

# Evaluate on the test set once, after model selection
test_error = loss(y_test, final_model.predict(x_test))
print(f'Final test error: {test_error:.4f}')

### 3.3 Why Do We Need Cross-Validation?

- A single validation split can be sensitive to the particular random split.
- Repeating training and validation across multiple folds gives us a more reliable estimate of validation performance.
- Cross-validation reduces our dependence on one particular validation split.

### 3.4 How Do We Perform Cross-Validation?

![Five-fold cross-validation with test set](https://raw.githubusercontent.com/mikemikezhu/COMP551_McGill/main/assets/five-fold-cross-validation.svg)

In [ ]:
def cross_validate(n, n_folds=5):
    """
    Generator function for K-fold cross-validation splits.
    
    Divides n samples into n_folds non-overlapping folds. In each iteration,
    one fold is used for validation and the remaining folds for training.
    
    Parameters:
    n: total number of samples to split
    n_folds: number of folds (default 5)
    
    Yields:
    tr_inds: list of training indices
    val_inds: list of validation indices
    
    Example with n=100, n_folds=5:
    - Fold 0: train on [20-99], validate on [0-19]
    - Fold 1: train on [0-19, 40-99], validate on [20-39]
    - ...
    - Fold 4: train on [0-79], validate on [80-99]
    """
    # Calculate validation set size for each fold
    # Integer division ensures even split (last fold may be slightly larger)
    n_val = n // n_folds
    
    # Note: This line creates empty list (overwriting unused permutation)
    # The original line 'inds = np.random.permutation(n)' is not used
    inds = []
    
    # Iterate through each fold
    for f in range(n_folds):
        # Initialize empty training indices list
        tr_inds = []
        
        # Calculate validation indices for this fold
        # Fold f normally uses [f*n_val, (f+1)*n_val); the final fold ends at n
        # Example: fold 2 with n_val=20 uses indices [40, 60)
        val_end = n if f == n_folds - 1 else (f+1)*n_val
        val_inds = list(range(f * n_val, val_end))
        
        # Build training indices from all OTHER folds
        # Training set = all indices except current validation fold
        
        # Add indices BEFORE current fold (folds 0 to f-1)
        if f > 0:
            # list(range(f*n_val)) creates [0, 1, ..., f*n_val-1]
            # These are all indices before the current validation fold
            tr_inds = list(range(f*n_val))
        
        # Add indices AFTER current fold (folds f+1 to n_folds-1)
        if f < n_folds - 1:
            # list(range((f+1)*n_val, n)) creates [(f+1)*n_val, ..., n-1]
            # Concatenate with existing tr_inds using + operator
            # These are all indices after the current validation fold
            tr_inds = tr_inds + list(range((f+1)*n_val, n))
        
        # The yield statement makes this a generator function
        # It suspends execution and returns (tr_inds, val_inds)
        # Next iteration resumes from here with the next fold
        # This is memory efficient: doesn't create all splits at once
        yield tr_inds, val_inds

In [ ]:
# Create random permutation of all data indices
# This shuffles the data before splitting
# inds shape: (num_instances,)
inds = np.random.permutation(num_instances)

# Separate the test data (first n_test samples after shuffling)
# Advanced indexing: inds[:n_test] selects first n_test shuffled indices
# x_test shape: (n_test, num_features)
# y_test shape: (n_test,)
x_test, y_test = x[inds[:n_test]], y[inds[:n_test]]

# Keep remaining data for training and cross-validation
# inds[n_test:] selects all indices after n_test
# x_rest shape: (num_instances - n_test, num_features)
# y_rest shape: (num_instances - n_test,)
x_rest, y_rest = x[inds[n_test:]], y[inds[n_test:]]

# Calculate size of remaining data (for cross-validation)
# This will be split into n_folds during cross-validation
n_rest = num_instances - n_test

In [ ]:
# Perform cross-validation to find the best K with error estimates
num_folds = 5

# Store one validation error per K per fold
# Shape: (number of candidate K values, number of folds)
err_valid = np.zeros((len(K_list), num_folds))

# Outer loop: iterate over different K values (hyperparameter search)
for i, K in enumerate(K_list):
    # Inner loop: iterate over cross-validation folds
    # enumerate() gives both fold index (f) and the split (tr, val)
    # cross_validate() is a generator that yields training/validation indices
    for f, (tr, val) in enumerate(cross_validate(n_rest, num_folds)):
        # Create KNN model with current K
        model = neighbors.KNeighborsRegressor(n_neighbors=K)
        
        # Train on training fold using advanced indexing
        # x_rest[tr] selects rows at indices in tr list
        # tr is a list of indices, x_rest[tr] selects those rows
        # Shape: x_rest[tr] is (len(tr), num_features)
        model = model.fit(x_rest[tr], y_rest[tr])
        
        # Evaluate on validation fold
        # val is a list of indices for validation samples
        # x_rest[val] selects validation samples, shape (len(val), num_features)
        # Store error in 2D array: row i is K value, column f is fold number
        err_valid[i, f] = loss(y_rest[val], model.predict(x_rest[val]))

# Average the validation errors across folds for each K
mean_valid_error = np.mean(err_valid, axis=1)

# Select the K with the lowest mean validation error
best_index = np.argmin(mean_valid_error)
best_K = K_list[best_index]
print(f'Best K: {best_K}')
print(f'Mean cross-validation error: {mean_valid_error[best_index]:.4f}')

# Refit one final model on all non-test data using the selected K
final_model = neighbors.KNeighborsRegressor(n_neighbors=best_K)
final_model.fit(x_rest, y_rest)

# Evaluate on the test set only once, after model selection is complete
test_error = loss(y_test, final_model.predict(x_test))
print(f'Final test error: {test_error:.4f}')

## 4. How Do We Evaluate the Final Model?

So far, we have used a regression example to study generalization and model selection. Now, let's switch to a classification problem and look at a useful tool for understanding a classifier's errors: the confusion matrix.

A confusion matrix shows how predictions compare with the true labels. Rows represent the true classes, columns represent the predicted classes, diagonal entries are correct predictions, and off-diagonal entries are classification errors.

In [ ]:
from sklearn import tree, model_selection

# Load the Iris dataset for classification
# x: features (sepal/petal dimensions), shape (150, 4)
# y: class labels (0, 1, or 2), shape (150,)
x, y = datasets.load_iris(return_X_y=True)

# Split into train (80%) and test (20%) sets
# test_size=0.2 means 20% of data goes to test set
# This function randomly shuffles and splits the data
# Returns: x_train, x_test, y_train, y_test
x_train, x_test, y_train, y_test = model_selection.train_test_split(x, y, test_size=0.2)

# Train a decision tree classifier and make predictions
# Method chaining: .fit() returns fitted model, .predict() returns predictions
# yh_test: predicted class labels for test set, shape (30,) approximately
yh_test = tree.DecisionTreeClassifier().fit(x_train, y_train).predict(x_test)

In [ ]:
def confusion_matrix(y, yh):
    """
    Compute confusion matrix for multi-class classification.
    
    The confusion matrix shows how predictions compare to true labels.
    Rows represent true classes, columns represent predicted classes.
    Element [i,j] = number of samples with true class i predicted as class j.
    
    Parameters:
    y: true class labels, shape (N,)
    yh: predicted class labels, shape (N,)
    
    Returns:
    c_matrix: confusion matrix, shape (n_classes, n_classes)
    
    Example for binary classification:
                 Predicted
                 0    1
    True  0  [[TN   FP]
          1   [FN   TP]]
    """
    # Calculate number of classes
    # np.max(y) + 1 assumes classes are labeled 0, 1, ..., K-1
    n_classes = np.max(y) + 1
    
    # Initialize confusion matrix with zeros
    # Shape: (n_classes, n_classes)
    c_matrix = np.zeros((n_classes, n_classes))
    
    # Nested loops to count predictions for each (true_class, predicted_class) pair
    for c1 in range(n_classes):
        for c2 in range(n_classes):
            # Boolean arithmetic to count matching conditions
            # (y==c1) creates boolean array: True where true label is c1, shape (N,)
            # (yh==c2) creates boolean array: True where prediction is c2, shape (N,)
            # * operator: element-wise multiplication (AND operation)
            #   Result is 1 when both conditions true, 0 otherwise
            # np.sum() counts total number of True values (sum of 1s)
            # This counts: samples with true class c1 predicted as c2
            c_matrix[c1, c2] = np.sum((y==c1)*(yh==c2))
    
    return c_matrix

# Compute and display confusion matrix
cmat = confusion_matrix(y_test, yh_test)
print(cmat)

For binary classification, the four entries of the confusion matrix are:

- True positive (TP)
- True negative (TN)
- False positive (FP)
- False negative (FN)

Metrics such as accuracy, precision, recall, false-positive rate, and false-discovery rate can be calculated from these values.

## 5. Summary
- Generalization: a model's capability to perform well on unseen samples sampled from the same underlying data distribution.
- The role of each dataset
    - **Training set:** Train the model.
    - **Validation set:** Select the model and hyperparameters.
    - **Test set:** Evaluate the final model; it should not be used for model selection.
- How to select a hyperparameter using validation set.
- Cross-validation
- Confusion matrix